# Tim — ViT + LoRA (strict cleaned workflow)

**Owner:** Tim only (avoid merge conflicts).

This notebook trains a **multi-label** movie-genre classifier from poster images using:

- strict cleaned dataset + strict splits
- pretrained ViT backbone
- LoRA adapters for parameter-efficient fine-tuning
- shared metrics JSON schema for team comparison

Default shared constants:
- image size: `224`
- inference threshold: `0.5`

In [1]:
from pathlib import Path

for p in Path("/content/drive/MyDrive").rglob("shared.py"):
    if p.parts[-2] == "experimentation":
        root = p.parent.parent
        if (root / "cleaned" / "MovieGenre_clean_with_images_full.csv").is_file():
            print("VALID ROOT:", root)
            break

VALID ROOT: /content/drive/MyDrive/ieor142b


In [2]:
# Run this first in Colab (Drive-synced repo + fast post-restart sync)
from google.colab import drive
from pathlib import Path
import os
import shutil

# Sync knobs:
# - USE_RUNTIME_COPY=True: train from /content (faster, preserves checkpoints across kernel restarts in same runtime)
# - FORCE_REFRESH=True: one-time full reset of runtime folder
# - SYNC_MODE="code_only": update only code/metadata files from Drive; skip heavy image recopy
USE_RUNTIME_COPY = True
FORCE_REFRESH = True   # MUST be True for first run after this fix — copies images to local disk
SYNC_MODE = "code_only"  # "code_only" or "full_copy" (images are always preserved on re-runs)

# 1) Mount Drive
drive.mount("/content/drive")

# 2) Resolve Drive repo path
DRIVE_REPO = Path("/content/drive/MyDrive/ieor142b")
if not DRIVE_REPO.exists():
    print(f"Default DRIVE_REPO not found: {DRIVE_REPO}")
    print("Searching MyDrive for ieor142b repo candidates...")

    candidates = []
    search_root = Path("/content/drive/MyDrive")
    if search_root.exists():
        for shared_file in search_root.rglob("shared.py"):
            if shared_file.parts[-2] != "experimentation":
                continue
            root = shared_file.parent.parent
            strict_csv = root / "cleaned" / "MovieGenre_clean_with_images_full.csv"
            if strict_csv.is_file():
                candidates.append(root)

    if not candidates:
        raise RuntimeError("Could not auto-find ieor142b under MyDrive. Set DRIVE_REPO manually.")

    DRIVE_REPO = candidates[0]
    print("Using discovered DRIVE_REPO:", DRIVE_REPO)

RUNTIME_REPO = Path("/content/ieor142b_runtime")


def copy_item(src: Path, dst: Path) -> None:
    if not src.exists():
        return
    if src.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)


if USE_RUNTIME_COPY:
    if FORCE_REFRESH and RUNTIME_REPO.exists():
        shutil.rmtree(RUNTIME_REPO)

    if not RUNTIME_REPO.exists():
        print("Creating initial runtime copy (code + images -> local /content, one-time setup)...")
        RUNTIME_REPO.mkdir(parents=True, exist_ok=True)

        # Copy code + metadata + strict CSV/splits.
        paths_to_copy = [
            "experimentation",
            "scripts",
            "results",
            "splits_cleaned",
            "README.md",
            "TEAM_DATA_SETUP.md",
            "cleaned/MovieGenre_clean_with_images_full.csv",
            "cleaned/MovieGenre_clean_with_images_full_manifest.json",
            "cleaned/README.md",
        ]
        for rel in paths_to_copy:
            copy_item(DRIVE_REPO / rel, RUNTIME_REPO / rel)

        # Copy images to local /content for fast disk I/O during training.
        drive_images = DRIVE_REPO / "cleaned" / "downloaded_posters"
        runtime_images = RUNTIME_REPO / "cleaned" / "downloaded_posters"
        runtime_images.parent.mkdir(parents=True, exist_ok=True)
        if runtime_images.exists() or runtime_images.is_symlink():
            if runtime_images.is_symlink():
                runtime_images.unlink()
            elif runtime_images.is_dir():
                shutil.rmtree(runtime_images)
        if drive_images.exists():
            print(f"Copying poster images to local /content (one-time, may take 5-10 min)...")
            shutil.copytree(drive_images, runtime_images)
            n_imgs = sum(1 for _ in runtime_images.glob("*.jpg"))
            print(f"Copied {n_imgs} poster images to local runtime.")
        else:
            print("WARNING: Drive poster images not found. Images will fall back to grey placeholder.")
    else:
        print("Reusing existing runtime copy:", RUNTIME_REPO)

        # Fast incremental sync for post-restart or code edits.
        if SYNC_MODE == "code_only":
            print("Sync mode: code_only (preserving checkpoints and image link/cache)")
            paths_to_sync = [
                "experimentation",
                "scripts",
                "results",
                "splits_cleaned",
                "README.md",
                "TEAM_DATA_SETUP.md",
                "cleaned/MovieGenre_clean_with_images_full.csv",
                "cleaned/MovieGenre_clean_with_images_full_manifest.json",
                "cleaned/README.md",
            ]
            for rel in paths_to_sync:
                copy_item(DRIVE_REPO / rel, RUNTIME_REPO / rel)
        else:
            print("Sync mode: full_copy")
            if RUNTIME_REPO.exists():
                shutil.rmtree(RUNTIME_REPO)
            shutil.copytree(DRIVE_REPO, RUNTIME_REPO)

    ACTIVE_REPO = RUNTIME_REPO
else:
    ACTIVE_REPO = DRIVE_REPO
    print("Using Drive repo directly:", ACTIVE_REPO)

os.chdir(str(ACTIVE_REPO))

# 3) Install notebook dependencies (idempotent)
!pip -q uninstall -y torchao || true
!pip -q install timm peft scikit-learn pandas pillow matplotlib

# 4) Ensure strict splits exist at ACTIVE_REPO
strict_csv = ACTIVE_REPO / "cleaned" / "MovieGenre_clean_with_images_full.csv"
strict_splits_dir = ACTIVE_REPO / "splits_cleaned"
required_split_files = [
    strict_splits_dir / "train_rows.csv",
    strict_splits_dir / "val_rows.csv",
    strict_splits_dir / "test_rows.csv",
]

print("Split files before check:", [p.is_file() for p in required_split_files])
if not all(p.is_file() for p in required_split_files):
    print("splits_cleaned missing; regenerating from strict CSV at ACTIVE_REPO...")
    !python scripts/generate_splits.py --csv-path "{strict_csv}" --out-dir "{strict_splits_dir}" --seed 42
print("Split files after check:", [p.is_file() for p in required_split_files])

# 5) Export root paths used by downstream cells
ROOT_OVERRIDE = str(ACTIVE_REPO)
os.environ["IEOR142B_ROOT"] = ROOT_OVERRIDE
os.environ["IEOR142B_DRIVE_REPO"] = str(DRIVE_REPO)

root = Path(ROOT_OVERRIDE)
print("Active repo root:", root)
print("Drive repo root:", DRIVE_REPO)
print("Exists:", root.exists())
print("shared.py:", (root / "experimentation" / "shared.py").is_file())
print("strict csv:", (root / "cleaned" / "MovieGenre_clean_with_images_full.csv").is_file())
print("strict train split:", (root / "splits_cleaned" / "train_rows.csv").is_file())
print("checkpoints dir:", (root / "checkpoints").resolve())

# Optional: if poster cache is missing in Drive, fetch it once.
# !python scripts/fetch_cleaned_images.py --verify
# !python scripts/fetch_cleaned_images.py

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Creating initial runtime copy (code + images -> local /content, one-time setup)...
Copying poster images to local /content (one-time, may take 5-10 min)...
Copied 9547 poster images to local runtime.
Split files before check: [True, True, True]
Split files after check: [True, True, True]
Active repo root: /content/ieor142b_runtime
Drive repo root: /content/drive/MyDrive/ieor142b
Exists: True
shared.py: True
strict csv: True
strict train split: True
checkpoints dir: /content/ieor142b_runtime/checkpoints


In [3]:
# 1) Imports + config

import json
import math
import os
import random
import sys
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Uses path from bootstrap cell (or manually set IEOR142B_ROOT in environment).
ROOT_OVERRIDE = os.environ.get("IEOR142B_ROOT")

def find_project_root(root_override: str | None = None) -> Path:
    """Locate ieor142b repo root across local/Colab setups."""

    def has_shared(base: Path) -> bool:
        return (base / "experimentation" / "shared.py").is_file()

    def has_strict_csv(base: Path) -> bool:
        return (base / "cleaned" / "MovieGenre_clean_with_images_full.csv").is_file()

    candidates: List[Path] = []

    override = root_override or os.environ.get("IEOR142B_ROOT", "")
    if override:
        candidates.append(Path(override).expanduser().resolve())

    here = Path.cwd().resolve()
    candidates.extend([here, *here.parents])

    # Common Colab locations.
    candidates.extend([
        Path("/content/ieor142b"),
        Path("/content/drive/MyDrive/ieor142b"),
        Path("/content/drive/MyDrive/Berkeley/IEOR_142B/proj/ieor142b"),
    ])

    seen = set()
    for base in candidates:
        if str(base) in seen:
            continue
        seen.add(str(base))
        if has_shared(base) and has_strict_csv(base):
            return base

    # Fallback: recursively search likely Colab roots for experimentation/shared.py
    search_roots = [Path("/content"), Path("/content/drive/MyDrive")]
    for sroot in search_roots:
        if not sroot.exists():
            continue
        try:
            for shared_file in sroot.rglob("shared.py"):
                if shared_file.parts[-3:] != ("experimentation", "shared.py"):
                    continue
                base = shared_file.parent.parent
                if has_shared(base) and has_strict_csv(base):
                    return base
        except Exception:
            # Permission errors or transient FS issues: continue searching other roots.
            pass

    msg = (
        "Could not find project root with strict cleaned dataset.\n"
        "Fix options:\n"
        "1) Set ROOT_OVERRIDE to your ieor142b absolute path.\n"
        "2) Or set env var IEOR142B_ROOT before running this cell.\n"
        "3) If using Drive, mount first: from google.colab import drive; drive.mount('/content/drive')\n"
        "4) Ensure these files exist under the same root:\n"
        "   - experimentation/shared.py\n"
        "   - cleaned/MovieGenre_clean_with_images_full.csv"
    )
    raise RuntimeError(msg)


# Optional manual override if auto-detection fails, e.g.:
# ROOT_OVERRIDE = "/content/ieor142b"
# If you set ROOT_OVERRIDE above, it will be used; otherwise env auto-detect applies.

_ROOT = find_project_root(ROOT_OVERRIDE)
sys.path.insert(0, str(_ROOT / "experimentation"))
print("Detected project root:", _ROOT)

from shared import (
    STRICT_BASELINE,
    STRICT_POSTER_DIR,
    load_strict_cleaned_dataframe,
    load_strict_split_dataframes,
    save_metrics_json,
    set_seed,
)


@dataclass
class RunConfig:
    # Default to full so accidental smoke runs are less likely.
    run_tier: str = "full"  # smoke or full
    model_name: str = "vit_base_patch16_224"
    threshold: float = 0.5
    img_size: int = 224
    num_workers: int = 2      # safe with local /content images
    pin_memory: bool = True   # faster GPU transfer when data is on local disk
    grad_clip_norm: float = 1.0
    early_stopping_patience: int = 4
    max_wallclock_minutes: int = 330
    use_focal_loss: bool = False
    lora_r: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.1
    batch_log_interval: int = 100


CFG = RunConfig()
if CFG.run_tier == "smoke":
    CFG.model_name = "vit_small_patch16_224"
    CFG.max_wallclock_minutes = 35

SEED = STRICT_BASELINE["seed"]
set_seed(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Project root:", _ROOT)
print("Device:", DEVICE)
print("Strict poster dir:", STRICT_POSTER_DIR)
print("Run tier:", CFG.run_tier)
print("Model:", CFG.model_name)

Detected project root: /content/ieor142b_runtime
Project root: /content/ieor142b_runtime
Device: cuda
Strict poster dir: /content/ieor142b_runtime/cleaned/downloaded_posters
Run tier: full
Model: vit_base_patch16_224


In [4]:
# 2) Data loading + dataloaders

if CFG.run_tier == "smoke":
    epochs = 3
    batch_size = 32 if DEVICE == "cuda" else 8
    train_frac = 0.15
    val_cap = 1000
else:
    epochs = 16
    batch_size = 32 if DEVICE == "cuda" else 8   # 32 fits T4 comfortably, halves batch count
    train_frac = 1.0
    val_cap = None

print(f"RUN MODE: {CFG.run_tier.upper()}")
if CFG.run_tier == "smoke":
    print(f"Smoke sampling active -> train_frac={train_frac}, val_cap={val_cap}")
else:
    print("Full-data training active -> no train/val subsampling")

strict_df = load_strict_cleaned_dataframe()
train_df, val_df, test_df, mlb = load_strict_split_dataframes()
num_genres = len(mlb.classes_)

if train_frac < 1.0:
    train_df = train_df.sample(frac=train_frac, random_state=SEED).reset_index(drop=True)
if val_cap is not None and len(val_df) > val_cap:
    val_df = val_df.sample(n=val_cap, random_state=SEED).reset_index(drop=True)

print("Strict full cleaned rows:", len(strict_df))
print("Train / Val / Test:", len(train_df), len(val_df), len(test_df), "| classes:", num_genres)


class PosterDataset(Dataset):
    def __init__(self, df: pd.DataFrame, mlb, transform, fallback_size: int = 224):
        self.df = df.reset_index(drop=True)
        self.mlb = mlb
        self.transform = transform
        self.fallback_size = fallback_size
        self.bad_images = 0

        # Verify images are on local disk (not Drive symlink) to catch I/O bottleneck early.
        sample_path = _ROOT / "cleaned" / "downloaded_posters"
        if sample_path.is_symlink():
            import warnings
            warnings.warn(
                f"downloaded_posters is a SYMLINK at {sample_path}. "
                "Images are still loading from Drive — this will be very slow (~11s/batch). "
                "Re-run bootstrap cell with FORCE_REFRESH=True to copy images locally."
            )
        elif not sample_path.exists():
            import warnings
            warnings.warn(f"downloaded_posters directory not found at {sample_path}. All images will use grey fallback.")

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img_path = row.get("image_path", "")
        if not isinstance(img_path, str) or not img_path:
            img_path = f"cleaned/downloaded_posters/{int(row['imdbId'])}.jpg"

        full_path = (_ROOT / img_path).resolve()
        try:
            image = Image.open(full_path).convert("RGB")
        except Exception:
            self.bad_images += 1
            image = Image.new("RGB", (self.fallback_size, self.fallback_size), (128, 128, 128))

        x = self.transform(image)
        genres = str(row["Genre"]).split("|")
        y = torch.tensor(self.mlb.transform([genres])[0], dtype=torch.float32)
        return x, y


mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(CFG.img_size, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(CFG.img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_ds = PosterDataset(train_df, mlb, train_tf, CFG.img_size)
val_ds = PosterDataset(val_df, mlb, val_tf, CFG.img_size)
test_ds = PosterDataset(test_df, mlb, val_tf, CFG.img_size)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=CFG.num_workers, pin_memory=CFG.pin_memory)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=CFG.pin_memory)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=CFG.pin_memory)

RUN MODE: FULL
Full-data training active -> no train/val subsampling
Strict full cleaned rows: 10875
Train / Val / Test: 8700 1087 1088 | classes: 24


In [5]:
# 2b) DataLoader diagnostics (run before training)
# TARGET: <2s per batch with local /content images on Colab T4.
# If batches take >10s each, images are still loading from Drive (symlink issue).

import time

def probe_loader(loader, name: str, n_batches: int = 10):
    print(f"\nProbing {name} loader for {n_batches} batches...")
    t0 = time.time()
    it = iter(loader)
    last = t0

    for i in range(1, n_batches + 1):
        x, y = next(it)
        now = time.time()
        dt = now - last
        flag = " <-- SLOW (Drive?)" if dt > 5.0 else ""
        print(
            f"  batch {i:02d}/{n_batches} | x={tuple(x.shape)} y={tuple(y.shape)} "
            f"| dt={dt:.2f}s | total={now-t0:.2f}s{flag}"
        )
        last = now

    elapsed = time.time() - t0
    per_batch = elapsed / n_batches
    print(f"{name} probe complete in {elapsed:.2f}s  ({per_batch:.2f}s/batch)")
    return elapsed, per_batch

train_probe_seconds, train_per_batch = probe_loader(train_loader, "train", n_batches=10)
val_probe_seconds, val_per_batch = probe_loader(val_loader, "val", n_batches=5)

# Estimate full epoch time.
n_train_batches = len(train_loader)
est_epoch_min = (n_train_batches * train_per_batch) / 60
print(f"\nEstimated epoch time: {n_train_batches} batches * {train_per_batch:.2f}s = {est_epoch_min:.1f} min/epoch")

# HARD FAIL: >10s/batch means Drive symlink is still active — abort before wasting GPU.
if train_per_batch > 10.0:
    raise RuntimeError(
        f"Images are loading from Drive (avg {train_per_batch:.1f}s/batch — expected <2s/batch). "
        "Re-run the bootstrap cell with FORCE_REFRESH=True to copy images to local /content. "
        "Training at this speed would take ~{est_epoch_min:.0f} min/epoch ({16*est_epoch_min:.0f} min total)."
    )

if train_per_batch > 3.0:
    print(f"WARNING: Loading is slower than expected ({train_per_batch:.1f}s/batch). "
          "Consider FORCE_REFRESH=True if this is >3s consistently.")

print("\nDataLoader diagnostics passed. Proceed to model/training cells.")


Probing train loader for 10 batches...
  batch 01/10 | x=(32, 3, 224, 224) y=(32, 24) | dt=1.35s | total=1.35s
  batch 02/10 | x=(32, 3, 224, 224) y=(32, 24) | dt=0.13s | total=1.48s
  batch 03/10 | x=(32, 3, 224, 224) y=(32, 24) | dt=0.03s | total=1.51s
  batch 04/10 | x=(32, 3, 224, 224) y=(32, 24) | dt=0.04s | total=1.55s
  batch 05/10 | x=(32, 3, 224, 224) y=(32, 24) | dt=0.51s | total=2.06s
  batch 06/10 | x=(32, 3, 224, 224) y=(32, 24) | dt=0.00s | total=2.06s
  batch 07/10 | x=(32, 3, 224, 224) y=(32, 24) | dt=0.69s | total=2.75s
  batch 08/10 | x=(32, 3, 224, 224) y=(32, 24) | dt=0.00s | total=2.75s
  batch 09/10 | x=(32, 3, 224, 224) y=(32, 24) | dt=0.70s | total=3.45s
  batch 10/10 | x=(32, 3, 224, 224) y=(32, 24) | dt=0.00s | total=3.45s
train probe complete in 3.45s  (0.34s/batch)

Probing val loader for 5 batches...
  batch 01/5 | x=(32, 3, 224, 224) y=(32, 24) | dt=0.54s | total=0.54s
  batch 02/5 | x=(32, 3, 224, 224) y=(32, 24) | dt=0.00s | total=0.54s
  batch 03/5 | x

In [6]:
# 3) Losses + metrics helpers

def compute_pos_weight(df: pd.DataFrame, mlb, max_weight: float = 5.0) -> torch.Tensor:
    """
    BCE pos_weight = neg/pos ratio per class.
    Capped at max_weight to prevent rare classes from dominating with huge weights,
    which causes the model to over-predict positives (high recall, low precision).
    """
    labels = mlb.transform(df["Genre"].str.split("|").tolist())
    pos = labels.sum(axis=0).astype(np.float32)
    neg = (labels.shape[0] - pos).astype(np.float32)
    pos = np.clip(pos, 1.0, None)
    weights = np.clip(neg / pos, 1.0, max_weight)
    return torch.tensor(weights, dtype=torch.float32)


class FocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss = alpha_t * ((1 - p_t) ** self.gamma) * bce
        return loss.mean()


def compute_metrics(logits: torch.Tensor, targets: torch.Tensor, threshold: float = 0.5) -> Dict[str, float]:
    preds = (torch.sigmoid(logits) >= threshold).float()
    tp = (preds * targets).sum(dim=1)
    fp = (preds * (1.0 - targets)).sum(dim=1)
    fn = ((1.0 - preds) * targets).sum(dim=1)

    precision = (tp / (tp + fp + 1e-8)).mean().item()
    recall = (tp / (tp + fn + 1e-8)).mean().item()
    f1 = (2.0 * tp / (2.0 * tp + fp + fn + 1e-8)).mean().item()
    exact = (preds == targets).all(dim=1).float().mean().item()

    return {"precision": precision, "recall": recall, "f1": f1, "exact_match": exact}

In [7]:
# 4) Build ViT + LoRA model

import warnings
import subprocess
import sys

# Harmless in VSCode/remote kernels: Colab secret vault lookup warning from HF hub.
warnings.filterwarnings("ignore", category=UserWarning, module=r"huggingface_hub\.utils\._auth")

# Preflight: remove incompatible torchao before importing peft.
try:
    import importlib.metadata as importlib_metadata

    torchao_version = importlib_metadata.version("torchao")

    def _parse(v: str):
        parts = []
        for token in v.split("."):
            num = "".join(ch for ch in token if ch.isdigit())
            parts.append(int(num) if num else 0)
        while len(parts) < 3:
            parts.append(0)
        return tuple(parts[:3])

    if _parse(torchao_version) < (0, 16, 0):
        print(f"Removing incompatible torchao {torchao_version} ...")
        subprocess.run([sys.executable, "-m", "pip", "-q", "uninstall", "-y", "torchao"], check=False)
except importlib_metadata.PackageNotFoundError:
    pass
except Exception as e:
    print("torchao preflight check skipped:", e)

import timm
from peft import LoraConfig, TaskType, get_peft_model


def build_lora_vit(model_name: str, num_labels: int):
    model = timm.create_model(model_name, pretrained=True, num_classes=num_labels)
    # Important: keep task_type unset for timm ViT so PEFT uses generic forward passthrough.
    # FEATURE_EXTRACTION wraps with NLP-style kwargs (input_ids, attention_mask), which breaks vision forward.
    lora_cfg = LoraConfig(
        r=CFG.lora_r,
        lora_alpha=CFG.lora_alpha,
        lora_dropout=CFG.lora_dropout,
        target_modules=["qkv"],
        modules_to_save=["head"],
        bias="none",
    )

    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return model

In [8]:
# 5) Train/eval epoch function

def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_logits: List[torch.Tensor] = []
    all_targets: List[torch.Tensor] = []

    for batch_idx, (images, targets) in enumerate(loader, start=1):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
            logits = model(images)
            loss = criterion(logits, targets)

        if is_train:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip_norm)
            scaler.step(optimizer)
            scaler.update()

        total_loss += loss.item() * images.size(0)
        all_logits.append(logits.detach().cpu())
        all_targets.append(targets.detach().cpu())

        if is_train and (batch_idx % CFG.batch_log_interval == 0):
            print(f"  batch {batch_idx}/{len(loader)} | loss={loss.item():.4f}")

    avg_loss = total_loss / len(loader.dataset)
    metrics = compute_metrics(torch.cat(all_logits), torch.cat(all_targets), threshold=CFG.threshold)
    return avg_loss, metrics

In [9]:
# 6) Initialize model, loss, optimizer, scheduler, checkpoints

RESUME_IF_AVAILABLE = True

model = build_lora_vit(CFG.model_name, num_genres).to(DEVICE)

pos_weight = compute_pos_weight(train_df, mlb).to(DEVICE)
if CFG.use_focal_loss:
    criterion = FocalLoss(alpha=STRICT_BASELINE["focal_alpha"], gamma=STRICT_BASELINE["focal_gamma"])
else:
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.05)
total_steps = max(1, epochs * len(train_loader))
warmup_steps = max(1, int(0.1 * total_steps))

def lr_lambda(step: int):
    if step < warmup_steps:
        return float(step + 1) / float(warmup_steps)
    progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

ckpt_dir = _ROOT / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)
best_path = ckpt_dir / "tim_vit_lora_best.pt"
last_path = ckpt_dir / "tim_vit_lora_last.pt"
state_path = ckpt_dir / "tim_vit_lora_train_state.pt"

history = []
best_val_f1 = -1.0
best_val_loss = float("inf")   # early-stop on val_loss (more stable than F1@fixed_threshold)
best_epoch = 0
patience_counter = 0
start_epoch = 1
start = time.time()

if RESUME_IF_AVAILABLE and last_path.is_file() and state_path.is_file():
    print("Resuming from checkpoint:", last_path)
    model.load_state_dict(torch.load(last_path, map_location=DEVICE))

    train_state = torch.load(state_path, map_location=DEVICE)
    if "optimizer" in train_state:
        optimizer.load_state_dict(train_state["optimizer"])
    if "scheduler" in train_state:
        scheduler.load_state_dict(train_state["scheduler"])
    if "scaler" in train_state and scaler is not None:
        scaler.load_state_dict(train_state["scaler"])

    history = train_state.get("history", [])
    best_val_f1 = float(train_state.get("best_val_f1", best_val_f1))
    best_val_loss = float(train_state.get("best_val_loss", best_val_loss))
    best_epoch = int(train_state.get("best_epoch", best_epoch))
    patience_counter = int(train_state.get("patience_counter", patience_counter))
    start_epoch = int(train_state.get("epoch", 0)) + 1

    print(f"Resume state -> start_epoch={start_epoch}, best_val_loss={best_val_loss:.4f}, best_epoch={best_epoch}")
else:
    print("Starting fresh training run.")

trainable params: 313,368 || all params: 86,130,480 || trainable%: 0.3638
Starting fresh training run.


In [10]:
# 7) Training loop

for epoch in range(start_epoch, epochs + 1):
    elapsed_minutes = (time.time() - start) / 60.0
    if elapsed_minutes >= CFG.max_wallclock_minutes:
        print(f"Stopping due to wallclock cap ({CFG.max_wallclock_minutes} min).")
        break

    train_loss, train_metrics = run_epoch(model, train_loader, criterion, optimizer=optimizer, scaler=scaler)
    val_loss, val_metrics = run_epoch(model, val_loader, criterion)
    scheduler.step()

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_f1": train_metrics["f1"],
        "val_f1": val_metrics["f1"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_exact": val_metrics["exact_match"],
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
        f"val_f1={val_metrics['f1']:.4f} "
        f"val_prec={val_metrics['precision']:.4f} val_rec={val_metrics['recall']:.4f} "
        f"val_exact={val_metrics['exact_match']:.4f}"
    )

    torch.save(model.state_dict(), last_path)

    # Track best val_f1 for final reporting, but checkpoint/early-stop on val_loss.
    # val_loss is smoother than F1@fixed_threshold and better reflects learning progress.
    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), best_path)
    else:
        patience_counter += 1

    # Persist resume state every epoch to survive runtime interruptions.
    torch.save(
        {
            "epoch": epoch,
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict() if scaler is not None else None,
            "history": history,
            "best_val_f1": best_val_f1,
            "best_val_loss": best_val_loss,
            "best_epoch": best_epoch,
            "patience_counter": patience_counter,
        },
        state_path,
    )

    if patience_counter >= CFG.early_stopping_patience:
        print("Early stopping triggered.")
        break

  batch 100/272 | loss=0.8978
  batch 200/272 | loss=1.0011
Epoch 01 | train_loss=0.9121 val_loss=0.8985 val_f1=0.1982 val_prec=0.1250 val_rec=0.5564 val_exact=0.0000
  batch 100/272 | loss=0.8777
  batch 200/272 | loss=0.8588
Epoch 02 | train_loss=0.8846 val_loss=0.8633 val_f1=0.2040 val_prec=0.1321 val_rec=0.5319 val_exact=0.0000
  batch 100/272 | loss=0.8037
  batch 200/272 | loss=0.7889
Epoch 03 | train_loss=0.8394 val_loss=0.8134 val_f1=0.2219 val_prec=0.1525 val_rec=0.5057 val_exact=0.0000
  batch 100/272 | loss=0.7926
  batch 200/272 | loss=0.7451
Epoch 04 | train_loss=0.7832 val_loss=0.7563 val_f1=0.2515 val_prec=0.1957 val_rec=0.4601 val_exact=0.0000
  batch 100/272 | loss=0.6956
  batch 200/272 | loss=0.7358
Epoch 05 | train_loss=0.7289 val_loss=0.7053 val_f1=0.2860 val_prec=0.2517 val_rec=0.4325 val_exact=0.0037
  batch 100/272 | loss=0.7785
  batch 200/272 | loss=0.6549
Epoch 06 | train_loss=0.6876 val_loss=0.6710 val_f1=0.3040 val_prec=0.2824 val_rec=0.4193 val_exact=0.009

In [11]:
# 7b) Threshold calibration (no retraining needed)
# Sweeps global threshold on the val set to find the value that maximises F1.
# This is critical because BCE loss training does not guarantee threshold=0.5 is optimal.

model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.eval()

all_logits_val = []
all_targets_val = []
with torch.no_grad():
    for images, targets in val_loader:
        images = images.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
            logits = model(images)
        all_logits_val.append(logits.cpu())
        all_targets_val.append(targets.cpu())

logits_val = torch.cat(all_logits_val)
targets_val = torch.cat(all_targets_val)

print(f"{'Threshold':>10} | {'Precision':>10} | {'Recall':>10} | {'F1':>10}")
print("-" * 48)

best_thr = 0.5
best_thr_f1 = -1.0
for thr in [t / 100 for t in range(10, 91, 5)]:
    m = compute_metrics(logits_val, targets_val, threshold=thr)
    marker = " <-- best" if m["f1"] > best_thr_f1 else ""
    if m["f1"] > best_thr_f1:
        best_thr_f1 = m["f1"]
        best_thr = thr
    print(f"{thr:>10.2f} | {m['precision']:>10.4f} | {m['recall']:>10.4f} | {m['f1']:>10.4f}{marker}")

print(f"\nBest threshold: {best_thr:.2f}  (val F1={best_thr_f1:.4f} vs default 0.5 F1={compute_metrics(logits_val, targets_val, 0.5)['f1']:.4f})")
print(f"Using threshold={best_thr:.2f} for test evaluation below.")
CFG.threshold = best_thr

 Threshold |  Precision |     Recall |         F1
------------------------------------------------
      0.10 |     0.1280 |     0.9845 |     0.2237 <-- best
      0.15 |     0.1536 |     0.9453 |     0.2599 <-- best
      0.20 |     0.1879 |     0.8950 |     0.3038 <-- best
      0.25 |     0.2337 |     0.8447 |     0.3558 <-- best
      0.30 |     0.2725 |     0.7886 |     0.3915 <-- best
      0.35 |     0.3069 |     0.7389 |     0.4186 <-- best
      0.40 |     0.3443 |     0.6690 |     0.4373 <-- best
      0.45 |     0.3763 |     0.5955 |     0.4400 <-- best
      0.50 |     0.4137 |     0.5103 |     0.4300
      0.55 |     0.4391 |     0.4238 |     0.4020
      0.60 |     0.4317 |     0.3306 |     0.3459
      0.65 |     0.3482 |     0.2392 |     0.2634
      0.70 |     0.2755 |     0.1679 |     0.1955
      0.75 |     0.1872 |     0.1024 |     0.1255
      0.80 |     0.1099 |     0.0557 |     0.0711
      0.85 |     0.0396 |     0.0199 |     0.0256
      0.90 |     0.0101 |    

In [12]:
# 8) Final evaluation + save artifacts

import shutil

model.load_state_dict(torch.load(best_path, map_location=DEVICE))
_, test_metrics = run_epoch(model, test_loader, criterion)

print("\nBest epoch:", best_epoch)
print("Best val F1:", round(best_val_f1, 4))
print("Test metrics:", {k: round(v, 4) for k, v in test_metrics.items()})
print("Bad image fallbacks:", {
    "train": train_ds.bad_images,
    "val": val_ds.bad_images,
    "test": test_ds.bad_images,
})

metrics_payload = {
    "model_name": "vit_lora",
    "seed": SEED,
    "img_size": CFG.img_size,
    "train_size": len(train_df),
    "val_size": len(val_df),
    "test_size": len(test_df),
    "best_val_f1": float(best_val_f1),
    "test_f1": float(test_metrics["f1"]),
    "test_precision": float(test_metrics["precision"]),
    "test_recall": float(test_metrics["recall"]),
    "test_exact_match": float(test_metrics["exact_match"]),
    "num_epochs_run": int(len(history)),
    "notes": (
        f"tier={CFG.run_tier}; model={CFG.model_name}; lora_r={CFG.lora_r}; "
        f"lora_alpha={CFG.lora_alpha}; lora_dropout={CFG.lora_dropout}; "
        f"threshold={CFG.threshold}; focal={CFG.use_focal_loss}; "
        f"max_wallclock_minutes={CFG.max_wallclock_minutes}"
    ),
}

metrics_path = _ROOT / "results" / "tim_vit_metrics.json"
save_metrics_json(metrics_path, metrics_payload)

history_path = _ROOT / "results" / "tim_vit_history.json"
with open(history_path, "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)
    f.write("\n")

print("Saved metrics:", metrics_path)
print("Saved history:", history_path)
print("Saved best checkpoint:", best_path)
print("Saved last checkpoint:", last_path)

# 9) Sync artifacts to Drive repo so they persist even if training used runtime copy.
drive_repo_env = os.environ.get("IEOR142B_DRIVE_REPO", "")
if drive_repo_env:
    drive_repo = Path(drive_repo_env)
    drive_ckpt_dir = drive_repo / "checkpoints"
    drive_results_dir = drive_repo / "results"
    drive_ckpt_dir.mkdir(parents=True, exist_ok=True)
    drive_results_dir.mkdir(parents=True, exist_ok=True)

    shutil.copy2(best_path, drive_ckpt_dir / best_path.name)
    shutil.copy2(last_path, drive_ckpt_dir / last_path.name)
    shutil.copy2(metrics_path, drive_results_dir / metrics_path.name)
    shutil.copy2(history_path, drive_results_dir / history_path.name)

    print("Synced best checkpoint:", drive_ckpt_dir / best_path.name)
    print("Synced last checkpoint:", drive_ckpt_dir / last_path.name)
    print("Synced metrics:", drive_results_dir / metrics_path.name)
    print("Synced history:", drive_results_dir / history_path.name)
else:
    print("Drive sync skipped: IEOR142B_DRIVE_REPO not set.")


Best epoch: 16
Best val F1: 0.43
Test metrics: {'precision': 0.3686, 'recall': 0.5864, 'f1': 0.4314, 'exact_match': 0.0165}
Bad image fallbacks: {'train': 0, 'val': 0, 'test': 0}
Saved metrics: /content/ieor142b_runtime/results/tim/tim_vit_metrics.json
Saved history: /content/ieor142b_runtime/results/tim/tim_vit_history.json
Saved best checkpoint: /content/ieor142b_runtime/checkpoints/tim_vit_lora_best.pt
Saved last checkpoint: /content/ieor142b_runtime/checkpoints/tim_vit_lora_last.pt
Synced best checkpoint: /content/drive/MyDrive/ieor142b/checkpoints/tim_vit_lora_best.pt
Synced last checkpoint: /content/drive/MyDrive/ieor142b/checkpoints/tim_vit_lora_last.pt
Synced metrics: /content/drive/MyDrive/ieor142b/results/tim/tim_vit_metrics.json
Synced history: /content/drive/MyDrive/ieor142b/results/tim/tim_vit_history.json
